# Classifying Penguins with Keras

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

2026-05-13 14:33:12.697593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [3]:
#drop nan rows
penguins = penguins.dropna()
penguins.shape

(333, 8)

In [4]:
penguins = penguins.sample(frac=1, random_state = 42).reset_index(drop=True)

In [5]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3250.0,39.5,16.7,178.0,True,False
1,3675.0,50.9,17.9,196.0,True,False
2,4000.0,42.1,19.1,195.0,False,True
3,4850.0,46.6,14.2,210.0,True,False
4,4050.0,41.1,18.2,192.0,False,True
...,...,...,...,...,...,...
328,4750.0,49.6,15.0,216.0,False,True
329,3900.0,37.2,19.4,184.0,False,True
330,3200.0,39.7,17.7,193.0,True,False
331,3950.0,45.2,17.8,198.0,True,False


In [6]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.152778,0.269091,0.428571,0.101695,1.0,0.0
1,0.270833,0.683636,0.571429,0.406780,1.0,0.0
2,0.361111,0.363636,0.714286,0.389831,0.0,1.0
3,0.597222,0.527273,0.130952,0.644068,1.0,0.0
4,0.375000,0.327273,0.607143,0.338983,0.0,1.0
...,...,...,...,...,...,...
328,0.569444,0.636364,0.226190,0.745763,0.0,1.0
329,0.333333,0.185455,0.750000,0.203390,0.0,1.0
330,0.138889,0.276364,0.547619,0.355932,1.0,0.0
331,0.347222,0.476364,0.559524,0.440678,1.0,0.0


In [7]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object


array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,

In [8]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
#make sure the model activation is right for 3 categories, not sigmoid
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [9]:
model.summary()

Model: "penguin_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │            49 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            18 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 119 (476.00 B)

 Trainable params: 119 (476.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [11]:

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)
history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.3445 - loss: 1.0941 - val_accuracy: 0.4706 - val_loss: 1.0894
Epoch 2/100


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/keras/src/backend/tensorflow/nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3813 - loss: 1.0849 - val_accuracy: 0.3824 - val_loss: 1.0845
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4080 - loss: 1.0792 - val_accuracy: 0.3235 - val_loss: 1.0808
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4114 - loss: 1.0749 - val_accuracy: 0.3235 - val_loss: 1.0780
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4281 - loss: 1.0711 - val_accuracy: 0.4118 - val_loss: 1.0753
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4548 - loss: 1.0678 - val_accuracy: 0.3824 - val_loss: 1.0733
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4749 - loss: 1.0649 - val_accuracy: 0.5000 - val_loss: 1.0711
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5084 - loss: 1.0622 - val_accuracy: 0.4706 - val_loss: 1.0694
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.5318 - loss: 1.0598 - val_accuracy: 0.5588 - val_loss: 1.0676
Epoc

In [ ]:
model_logit_false = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_false.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_false.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_false.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 73ms/step - accuracy: 0.7926 - loss: 0.5320 - val_accuracy: 0.7941 - val_loss: 0.5003
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7893 - loss: 0.5152 - val_accuracy: 0.7941 - val_loss: 0.4875
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7960 - loss: 0.5034 - val_accuracy: 0.7941 - val_loss: 0.4760
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7893 - loss: 0.4937 - val_accuracy: 0.7941 - val_loss: 0.4674
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7926 - loss: 0.4850 - val_accuracy: 0.7941 - val_loss: 0.4584
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7960 - loss: 0.4773 - val_accuracy: 0.7941 - val_loss: 0.4497
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7960 - loss: 0.4696 - val_accuracy: 0.7941 - val_loss: 0.4419
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7960 - loss: 0.4624 - val_accuracy: 0.7941 - val_loss:

In [13]:
model_logit_false.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


array([[9.34798002e-01, 4.65603620e-02, 1.86416432e-02],
       [1.40357884e-02, 7.94725239e-01, 1.91238940e-01],
       [9.79492784e-01, 1.41025167e-02, 6.40468020e-03],
       [2.57308802e-05, 5.55673381e-03, 9.94417489e-01],
       [9.80454385e-01, 1.34193627e-02, 6.12621848e-03],
       [2.32068766e-02, 8.06900144e-01, 1.69892997e-01],
       [2.76306540e-01, 5.66590786e-01, 1.57102600e-01],
       [2.05340533e-04, 6.60905764e-02, 9.33704019e-01],
       [1.24865037e-05, 1.18632428e-02, 9.88124311e-01],
       [1.48922816e-04, 1.47480210e-02, 9.85103011e-01],
       [7.07616508e-01, 2.19066262e-01, 7.33170807e-02],
       [9.97629046e-01, 1.36810343e-03, 1.00282242e-03],
       [4.92099613e-01, 3.89296681e-01, 1.18603610e-01],
       [9.76954281e-01, 1.59106720e-02, 7.13492325e-03],
       [5.16414605e-02, 7.72298813e-01, 1.76059693e-01],
       [9.62233663e-01, 2.65052337e-02, 1.12610077e-02],
       [9.81745124e-01, 1.25040589e-02, 5.75073482e-03],
       [1.94537788e-05, 2.97242

In [14]:
penguins['species']

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object

In [15]:
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,